# Reinforcement Learning (Aprendizaje por Refuerzo)

Implementación desde cero de algoritmos de Reinforcement Learning: Q-Learning y SARSA para problemas de decisión secuencial.

## ¿Cuál es el objetivo del Reinforcement Learning?

Imagina que estás entrenando a un perro. No le puedes explicar con palabras qué debe hacer: simplemente le das una **galleta** (recompensa) cuando hace algo bien, y un **«no!»** (castigo) cuando hace algo mal. Con el tiempo, el perro aprende qué acciones llevan a galletas y cuáles a castigos. ¡Eso es Reinforcement Learning!

El **Aprendizaje por Refuerzo (RL)** es un paradigma de Machine Learning donde un **agente** aprende a tomar **decisiones secuenciales** interactuando con un **entorno**, con el objetivo de **maximizar una recompensa acumulada** a lo largo del tiempo.

### ¿En qué se diferencia de otros paradigmas?

| Característica | Aprendizaje Supervisado | Aprendizaje No Supervisado | Aprendizaje por Refuerzo |
|---|---|---|---|
| **Datos** | Pares (entrada, etiqueta) | Solo entradas | Experiencias (estado, acción, recompensa) |
| **Señal de aprendizaje** | Etiquetas correctas | Ninguna | Recompensas (escalares, retardadas) |
| **Objetivo** | Predecir etiquetas | Encontrar patrones/estructura | Maximizar recompensa acumulada |
| **Feedback** | Inmediato y directo | No hay feedback | Retardado y evaluativo |
| **Ejemplo** | Clasificar imágenes | Agrupar clientes | Jugar ajedrez |

La diferencia clave es que en RL:
- **No hay un «profesor»** que diga cuál es la acción correcta (a diferencia del supervisado).
- **Sí hay un objetivo claro** que optimizar (a diferencia del no supervisado).
- Las **acciones afectan estados futuros**: lo que haces ahora impacta lo que puedes hacer después.

### Conceptos clave

| Concepto | Descripción | Ejemplo (perro) |
|---|---|---|
| **Agente** | Quien toma las decisiones | El perro |
| **Entorno** | El mundo con el que interactúa | La casa, el parque |
| **Estado ($s$)** | Situación actual del entorno | Posición del perro, objetos cercanos |
| **Acción ($a$)** | Lo que el agente puede hacer | Sentarse, dar la pata, ladrar |
| **Recompensa ($r$)** | Señal numérica de feedback | +1 galleta, -1 regaño |
| **Política ($\pi$)** | Estrategia: qué acción tomar en cada estado | «Si el dueño dice 'sienta', sentárse» |

### El bucle de Reinforcement Learning

El proceso funciona como un ciclo repetitivo:

```
  ┌─────────┐       acción (a_t)        ┌───────────┐
  │  AGENTE  │ ───────────────────→ │  ENTORNO  │
  │ (política)│ ←─────────────────── │           │
  └─────────┘  estado (s_{t+1})   └───────────┘
                recompensa (r_{t+1})
```

1. El agente observa el **estado actual** $s_t$ del entorno.
2. Basado en su **política** $\pi$, elige una **acción** $a_t$.
3. El entorno responde con un **nuevo estado** $s_{t+1}$ y una **recompensa** $r_{t+1}$.
4. El agente usa esta experiencia para **mejorar su política**.
5. Se repite el ciclo.

### Aplicaciones en el mundo real

- **Juegos:** AlphaGo (Go), OpenAI Five (Dota 2), Atari. El agente aprende estrategias jugando millones de partidas.
- **Robótica:** Robots que aprenden a caminar, agarrar objetos o navegar sin programación explícita.
- **Sistemas de recomendación:** Decidir qué contenido mostrar para maximizar el engagement del usuario.
- **Conducción autónoma:** Aprender políticas de conducción segura mediante simulación.
- **Finanzas:** Estrategias de trading que se adaptan a condiciones cambiantes del mercado.
- **Control industrial:** Optimizar el consumo energético en centros de datos (Google DeepMind redujo el enfriamiento en un 40%).

## Un ejemplo sencillo

Para entender cómo funciona el RL en la práctica, usaremos un **GridWorld** (mundo de cuadrícula) de 4x4.

Las reglas son simples:
- El agente comienza en la esquina superior izquierda **(0, 0)**.
- Debe llegar a la meta en la esquina inferior derecha **(3, 3)**.
- Hay un **obstáculo** en la posición **(1, 1)** que debe evitar.
- Puede moverse en 4 direcciones: **arriba, abajo, izquierda, derecha**.
- Cada paso cuesta **-1** (para motivar caminos cortos).
- Pisar el obstáculo cuesta **-10**.
- Llegar a la meta da **+100**.

¿Cuál es el mejor camino? Eso es lo que el agente debe aprender.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Definir el GridWorld 4x4 ---
grid_size = 4
start = (0, 0)
goal = (3, 3)
obstacle = (1, 1)

# Estructura de recompensas
rewards = np.full((grid_size, grid_size), -1.0)  # -1 por cada paso
rewards[obstacle] = -10.0   # obstaculo
rewards[goal] = 100.0       # meta

# --- Visualizacion del GridWorld ---
fig, ax = plt.subplots(1, 1, figsize=(7, 7))

# Colores para cada tipo de celda
for i in range(grid_size):
    for j in range(grid_size):
        if (i, j) == start:
            color = '#90EE90'  # verde claro - inicio
            label = 'INICIO\n(0,0)'
        elif (i, j) == goal:
            color = '#FFD700'  # dorado - meta
            label = 'META\n(3,3)\n+100'
        elif (i, j) == obstacle:
            color = '#FF6B6B'  # rojo - obstaculo
            label = 'OBST.\n(1,1)\n-10'
        else:
            color = '#E8E8E8'  # gris claro - normal
            label = f'({i},{j})\n-1'

        rect = plt.Rectangle((j, grid_size - 1 - i), 1, 1,
                             facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(j + 0.5, grid_size - 1 - i + 0.5, label,
                ha='center', va='center', fontsize=10, fontweight='bold')

# Dibujar una flecha mostrando un posible camino
path = [(0,0), (0,1), (0,2), (1,2), (2,2), (2,3), (3,3)]
for k in range(len(path)-1):
    r1, c1 = path[k]
    r2, c2 = path[k+1]
    dx = (c2 - c1) * 0.6
    dy = (r1 - r2) * 0.6
    ax.annotate('', xy=(c2 + 0.5 - dx*0.15, grid_size - 1 - r2 + 0.5 - dy*0.15),
                xytext=(c1 + 0.5 + dx*0.15, grid_size - 1 - r1 + 0.5 + dy*0.15),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2.5, alpha=0.6))

ax.set_xlim(0, grid_size)
ax.set_ylim(0, grid_size)
ax.set_aspect('equal')
ax.set_title('GridWorld 4x4: El agente debe ir de INICIO a META', fontsize=13, fontweight='bold')
ax.axis('off')

# Leyenda
legend_elements = [
    mpatches.Patch(facecolor='#90EE90', edgecolor='black', label='Inicio'),
    mpatches.Patch(facecolor='#FFD700', edgecolor='black', label='Meta (+100)'),
    mpatches.Patch(facecolor='#FF6B6B', edgecolor='black', label='Obstaculo (-10)'),
    mpatches.Patch(facecolor='#E8E8E8', edgecolor='black', label='Celda normal (-1)')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

# --- Representacion en texto ---
print('\nRepresentacion del GridWorld en texto:')
print('=' * 37)
symbols = {start: ' S ', goal: ' G ', obstacle: ' X '}
for i in range(grid_size):
    row = '|'
    for j in range(grid_size):
        row += symbols.get((i, j), ' . ') + '|'
    print(row)
print('=' * 37)
print('S = Inicio, G = Meta, X = Obstaculo, . = Celda normal')
print(f'\nEstructura de recompensas:')
print(f'  Cada paso:   -1  (motiva caminos cortos)')
print(f'  Obstaculo:  -10  (penalizacion fuerte)')
print(f'  Meta:      +100  (recompensa alta)')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

grid_size = 4
start = (0, 0)
goal = (3, 3)
obstacle = (1, 1)

# Acciones: 0=arriba, 1=abajo, 2=izquierda, 3=derecha
action_names = ['arriba', 'abajo', 'izquierda', 'derecha']
action_arrows = {'arriba': '\u2191', 'abajo': '\u2193', 'izquierda': '\u2190', 'derecha': '\u2192'}
# Desplazamientos: (delta_fila, delta_columna)
action_deltas = {'arriba': (-1, 0), 'abajo': (1, 0), 'izquierda': (0, -1), 'derecha': (0, 1)}

# --- Politica aleatoria ---
np.random.seed(42)
random_policy = np.random.randint(0, 4, size=(grid_size, grid_size))

# --- Politica optima (disenada manualmente) ---
# Cada celda apunta hacia la meta evitando el obstaculo
optimal_policy = np.array([
    [1, 3, 1, 1],   # fila 0: abajo, derecha, abajo, abajo
    [3, 0, 1, 1],   # fila 1: derecha, (obstaculo), abajo, abajo
    [3, 3, 3, 1],   # fila 2: derecha, derecha, derecha, abajo
    [3, 3, 3, 0]    # fila 3: derecha, derecha, derecha, (meta)
])

def plot_policy(ax, policy, title):
    """Dibuja una politica como flechas en el grid."""
    for i in range(grid_size):
        for j in range(grid_size):
            if (i, j) == start:
                color = '#90EE90'
            elif (i, j) == goal:
                color = '#FFD700'
            elif (i, j) == obstacle:
                color = '#FF6B6B'
            else:
                color = '#E8E8E8'

            rect = plt.Rectangle((j, grid_size - 1 - i), 1, 1,
                                 facecolor=color, edgecolor='black', linewidth=2)
            ax.add_patch(rect)

            if (i, j) == goal:
                ax.text(j + 0.5, grid_size - 1 - i + 0.5, 'META',
                        ha='center', va='center', fontsize=11, fontweight='bold')
            elif (i, j) == obstacle:
                ax.text(j + 0.5, grid_size - 1 - i + 0.5, 'X',
                        ha='center', va='center', fontsize=16, fontweight='bold', color='white')
            else:
                arrow = action_arrows[action_names[policy[i, j]]]
                ax.text(j + 0.5, grid_size - 1 - i + 0.5, arrow,
                        ha='center', va='center', fontsize=22, fontweight='bold', color='#333333')

    ax.set_xlim(0, grid_size)
    ax.set_ylim(0, grid_size)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

# --- Comparacion lado a lado ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_policy(ax1, random_policy, 'Politica Aleatoria\n(el agente no sabe que hacer)')
plot_policy(ax2, optimal_policy, 'Politica Optima\n(camino mas corto a la meta)')

plt.suptitle('Comparacion: Politica Aleatoria vs. Politica Optima',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Politica Aleatoria (acciones elegidas al azar):')
for i in range(grid_size):
    row = '  |'
    for j in range(grid_size):
        if (i, j) == goal:
            row += ' G |'
        elif (i, j) == obstacle:
            row += ' X |'
        else:
            row += f' {action_arrows[action_names[random_policy[i,j]]]} |'
    print(row)

print('\nPolitica Optima (aprendida por el agente):')
for i in range(grid_size):
    row = '  |'
    for j in range(grid_size):
        if (i, j) == goal:
            row += ' G |'
        elif (i, j) == obstacle:
            row += ' X |'
        else:
            row += f' {action_arrows[action_names[optimal_policy[i,j]]]} |'
    print(row)

print('\nEl objetivo del RL es aprender la politica optima a partir de la experiencia.')

Pero... ¿cómo aprende el agente cuál es la mejor acción en cada estado?

No podemos simplemente decirle cuál es la política óptima (eso sería aprendizaje supervisado). El agente debe **descubrirla por sí mismo** interactuando con el entorno.

Para eso necesitamos un marco matemático riguroso que nos permita:
1. **Modelar** el problema de decisión secuencial.
2. **Definir** qué significa «ser óptimo».
3. **Derivar** algoritmos que converjan a la solución óptima.

## Demostración Matemática

### Paso 1: Proceso de Decisión de Markov (MDP)

El marco formal para RL es el **Proceso de Decisión de Markov (MDP)**, definido como una tupla:

$$
\text{MDP} = (S, A, P, R, \gamma)
$$

Donde:

- $S$: **Conjunto de estados.** Todas las situaciones posibles del entorno. En nuestro GridWorld 4x4, $S = \{(0,0), (0,1), \ldots, (3,3)\}$ con $|S| = 16$ estados.

- $A$: **Conjunto de acciones.** Todo lo que el agente puede hacer. En nuestro caso, $A = \{\text{arriba}, \text{abajo}, \text{izquierda}, \text{derecha}\}$ con $|A| = 4$ acciones.

- $P(s'|s, a)$: **Función de transición.** La probabilidad de llegar al estado $s'$ si estamos en el estado $s$ y tomamos la acción $a$. En un entorno determinístico (como nuestro GridWorld), $P(s'|s,a) = 1$ para un único $s'$ y $0$ para los demás.

- $R(s, a, s')$: **Función de recompensa.** La recompensa inmediata que recibe el agente al pasar del estado $s$ al estado $s'$ mediante la acción $a$. A veces se simplifica como $R(s, a)$ o $R(s)$.

- $\gamma \in [0, 1]$: **Factor de descuento.** Controla cuánto valora el agente las recompensas futuras respecto a las inmediatas.

#### La propiedad de Markov

La característica fundamental de un MDP es la **propiedad de Markov**: el futuro solo depende del estado presente, no del pasado.

$$
P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, \ldots, s_0, a_0) = P(s_{t+1} | s_t, a_t)
$$

Es decir, el estado actual contiene **toda la información relevante** para decidir qué pasará después. No importa cómo llegamos ahí.

En nuestro GridWorld: si el agente está en (2, 1), no importa si llegó desde (1, 1) o desde (2, 0). Lo único que importa es que ahora está en (2, 1).

### Paso 2: Retorno y factor de descuento

El agente no quiere maximizar solo la recompensa inmediata, sino la **recompensa acumulada a largo plazo**. Definimos el **retorno** $G_t$ como la suma de recompensas futuras descontadas:

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}
$$

#### ¿Por qué descontar las recompensas futuras?

1. **Preferencia temporal:** Una recompensa hoy vale más que la misma recompensa mañana (como el dinero: mejor $100 hoy que $100 en un año).

2. **Convergencia matemática:** Sin descuento ($\gamma = 1$), la suma podría ser infinita en episodios sin fin. Con $\gamma < 1$, la serie geométrica converge: $\sum_{k=0}^{\infty} \gamma^k = \frac{1}{1-\gamma}$.

3. **Incertidumbre:** El futuro lejano es más incierto, así que le damos menos peso.

#### Efecto de $\gamma$

- $\gamma = 0$: **Miópico.** Solo importa la recompensa inmediata $R_{t+1}$. El agente es «codicioso».
- $\gamma = 1$: **Previsor.** Todas las recompensas futuras valen lo mismo. Puede ser problemático si los episodios no terminan.
- $\gamma = 0.9$ (típico): Buen balance. Una recompensa dentro de 10 pasos vale $0.9^{10} \approx 0.35$ de su valor original.

#### Ejemplo numérico

Supongamos que un agente recibe recompensas $R_1 = -1, R_2 = -1, R_3 = -1, R_4 = +100$ (camino de 3 pasos hasta la meta), con $\gamma = 0.9$:

$$
G_0 = (-1) + 0.9(-1) + 0.9^2(-1) + 0.9^3(100)
$$
$$
G_0 = -1 - 0.9 - 0.81 + 72.9 = 70.19
$$

Si el camino fuera más largo (5 pasos): $R_1 = -1, R_2 = -1, R_3 = -1, R_4 = -1, R_5 = -1, R_6 = +100$:

$$
G_0 = (-1) + 0.9(-1) + 0.81(-1) + 0.729(-1) + 0.6561(-1) + 0.59049(100)
$$
$$
G_0 = -1 - 0.9 - 0.81 - 0.729 - 0.6561 + 59.049 = 54.9539
$$

El camino corto da $G_0 = 70.19$ vs el largo $G_0 = 54.95$. ¡El descuento incentiva naturalmente los caminos más cortos!

### Paso 3: Función de valor $V(s)$ y $Q(s, a)$

Para evaluar qué tan «bueno» es un estado o una acción, definimos dos funciones de valor.

#### Función de valor de estado $V^\pi(s)$

Es el retorno esperado comenzando en el estado $s$ y siguiendo la política $\pi$:

$$
V^\pi(s) = \mathbb{E}_\pi[G_t | S_t = s] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \bigg| S_t = s\right]
$$

Intuitivamente: «Si estoy en el estado $s$ y sigo la política $\pi$, ¿cuánta recompensa puedo esperar acumular?»

#### Función de valor de acción $Q^\pi(s, a)$

Es el retorno esperado comenzando en $s$, tomando la acción $a$, y después siguiendo $\pi$:

$$
Q^\pi(s, a) = \mathbb{E}_\pi[G_t | S_t = s, A_t = a] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \bigg| S_t = s, A_t = a\right]
$$

Intuitivamente: «Si estoy en $s$, tomo la acción $a$, y después sigo con $\pi$, ¿cuánta recompensa espero?»

#### Relación entre $V$ y $Q$

$$
V^\pi(s) = \sum_{a \in A} \pi(a|s) \, Q^\pi(s, a)
$$

Es decir, el valor de un estado es el **promedio ponderado** de los valores Q de todas las acciones, ponderado por la probabilidad de elegir cada acción según la política.

#### Funciones óptimas

Las funciones de valor **óptimas** son las mejores posibles sobre todas las políticas:

$$
V^*(s) = \max_\pi V^\pi(s) = \max_a Q^*(s, a)
$$

$$
Q^*(s, a) = \max_\pi Q^\pi(s, a)
$$

Si conocemos $Q^*$, la política óptima es simplemente elegir la acción con mayor valor Q:

$$
\pi^*(s) = \arg\max_a Q^*(s, a)
$$

### Paso 4: Ecuaciones de Bellman

Las **ecuaciones de Bellman** son la piedra angular de RL. Expresan una relación recursiva: el valor de un estado depende del valor de los estados siguientes.

#### Ecuación de Bellman de expectativa

Para una política $\pi$ fija:

$$
V^\pi(s) = \sum_{a} \pi(a|s) \left[ R(s, a) + \gamma \sum_{s'} P(s'|s, a) \, V^\pi(s') \right]
$$

Esta ecuación dice: el valor de estar en $s$ bajo $\pi$ es la suma sobre todas las acciones de:
- La probabilidad de tomar esa acción $\pi(a|s)$, multiplicada por
- La recompensa inmediata $R(s,a)$ más el valor descontado del siguiente estado.

#### Ecuación de Bellman de optimalidad

Para la política óptima:

$$
V^*(s) = \max_{a} \left[ R(s, a) + \gamma \sum_{s'} P(s'|s, a) \, V^*(s') \right]
$$

La diferencia es que en lugar de promediar sobre acciones, **elegimos la mejor** (max).

#### Forma Q de la ecuación de optimalidad

$$
Q^*(s, a) = R(s, a) + \gamma \sum_{s'} P(s'|s, a) \, \max_{a'} Q^*(s', a')
$$

Esta es especialmente importante porque es la base de **Q-Learning**.

#### Intuición

La idea fundamental es:

$$
\text{Valor} = \text{Recompensa inmediata} + \gamma \times \text{Valor futuro descontado}
$$

Es como pensar: «¿Cuánto vale este estado? Lo que gano ahora, más lo que ganaré después (descontado).»

### Paso 5: Q-Learning

Las ecuaciones de Bellman son elegantes, pero requieren conocer $P(s'|s,a)$ (el modelo del entorno). En la práctica, muchas veces **no conocemos** cómo funciona el entorno internamente.

**Q-Learning** es un algoritmo **model-free** (libre de modelo): aprende $Q^*$ directamente de la experiencia, sin necesidad de conocer las probabilidades de transición.

#### Regla de actualización

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]
$$

Donde:
- $\alpha \in (0, 1]$: **tasa de aprendizaje** (learning rate). Controla qué tanto actualizamos el valor Q.
- $R$: recompensa recibida al tomar la acción $a$ en el estado $s$.
- $s'$: el nuevo estado al que llegamos.
- $\max_{a'} Q(s', a')$: el mejor valor Q en el nuevo estado (sobre todas las acciones posibles).

#### El error de diferencia temporal (TD error)

El término entre corchetes se llama **error TD (Temporal Difference)**:

$$
\delta = R + \gamma \max_{a'} Q(s', a') - Q(s, a)
$$

- $R + \gamma \max_{a'} Q(s', a')$: es el **objetivo** (target) — lo que creemos que debería valer $Q(s,a)$ basado en la nueva experiencia.
- $Q(s, a)$: es nuestra **estimación actual**.
- $\delta$: es la **sorpresa** — la diferencia entre lo que observamos y lo que esperábamos.

Si $\delta > 0$: la experiencia fue **mejor** de lo esperado → aumentamos $Q(s,a)$.
Si $\delta < 0$: la experiencia fue **peor** de lo esperado → disminuimos $Q(s,a)$.
Si $\delta = 0$: la estimación era perfecta → no cambia nada.

#### Características de Q-Learning

- **Off-policy:** Aprende la política óptima ($\max$) independientemente de la política que use para explorar. Esto significa que puede explorar aleatoriamente y aún así aprender la solución óptima.
- **Convergencia garantizada:** Bajo ciertas condiciones (todos los pares estado-acción visitados infinitas veces, $\alpha$ decreciente apropiadamente), Q-Learning converge a $Q^*$.
- **Tabla Q:** En problemas discretos, almacenamos los valores Q en una tabla de $|S| \times |A|$ entradas.

### Paso 6: SARSA

**SARSA** es una alternativa **on-policy** a Q-Learning. El nombre viene de la tupla que usa: **S**tate, **A**ction, **R**eward, **S**tate, **A**ction.

#### Regla de actualización

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma Q(s', a') - Q(s, a) \right]
$$

#### Diferencia clave con Q-Learning

| Aspecto | Q-Learning | SARSA |
|---|---|---|
| **Usa** | $\max_{a'} Q(s', a')$ | $Q(s', a')$ (acción real tomada) |
| **Tipo** | Off-policy | On-policy |
| **Aprende** | Política óptima (independiente de exploración) | Política que realmente sigue |
| **Comportamiento** | Más agresivo, puede ser riesgoso | Más conservador, más seguro |

La diferencia fundamental:
- **Q-Learning** usa el **máximo** de Q en el siguiente estado. Asume que en el futuro siempre tomará la mejor acción.
- **SARSA** usa el valor Q de la **acción que realmente tomó** en el siguiente estado (que podría ser aleatoria por exploración).

#### ¿Cuándo es mejor SARSA?

SARSA es más conservador porque **tiene en cuenta que el agente a veces explora** (toma acciones aleatorias). Esto lo hace más seguro en entornos donde la exploración puede llevar a estados peligrosos.

Ejemplo clásico: un agente caminando al borde de un acantilado. Q-Learning aprenderá el camino más corto (junto al borde), pero SARSA aprenderá un camino más largo pero seguro (lejos del borde), porque sabe que a veces explorará y podría caer.

### Paso 7: Exploración vs Explotación

Uno de los dilemas más importantes en RL es el **balance entre exploración y explotación**:

- **Explotación:** Usar la mejor acción conocida para maximizar la recompensa ahora.
- **Exploración:** Probar acciones nuevas o aleatorias para descubrir mejores estrategias.

Si solo explotas, podrías quedarte con una estrategia subóptima (nunca descubrir algo mejor). Si solo exploras, nunca aprovechas lo que has aprendido.

#### Estrategia $\varepsilon$-greedy

La solución más común es **$\varepsilon$-greedy**:

$$
a_t = \begin{cases}
\text{acción aleatoria} & \text{con probabilidad } \varepsilon \\
\arg\max_a Q(s_t, a) & \text{con probabilidad } 1 - \varepsilon
\end{cases}
$$

- Con probabilidad $\varepsilon$: **explora** (acción aleatoria).
- Con probabilidad $1 - \varepsilon$: **explota** (mejor acción conocida).

#### Decaimiento de $\varepsilon$

Es común reducir $\varepsilon$ a lo largo del entrenamiento:

$$
\varepsilon_t = \max(\varepsilon_{\min}, \ \varepsilon_0 \cdot \text{decay}^t)
$$

- Al inicio: $\varepsilon$ alto (mucha exploración, el agente no sabe nada).
- Al final: $\varepsilon$ bajo (mucha explotación, el agente ya sabe qué hacer).

Valores típicos: $\varepsilon_0 = 1.0$, $\varepsilon_{\min} = 0.01$, decay $= 0.995$.

### Visualización del factor de descuento $\gamma$

Veamos cómo diferentes valores de $\gamma$ afectan el peso que se le da a las recompensas futuras.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Efecto de gamma en el peso de recompensas futuras ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Grafico 1: gamma^k para diferentes valores de gamma
steps = np.arange(0, 25)
gammas = [0.5, 0.9, 0.99]
colors = ['#e74c3c', '#3498db', '#2ecc71']

for gamma, color in zip(gammas, colors):
    weights = gamma ** steps
    ax1.plot(steps, weights, 'o-', color=color, label=f'$\\gamma = {gamma}$',
             linewidth=2, markersize=4)

ax1.set_xlabel('Pasos en el futuro (k)', fontsize=12)
ax1.set_ylabel('Peso $\\gamma^k$', fontsize=12)
ax1.set_title('Peso de las recompensas futuras segun $\\gamma$', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.05, 1.05)

# Anotaciones
ax1.annotate('$\\gamma=0.5$: recompensas lejanas\nvalen muy poco',
             xy=(5, 0.5**5), xytext=(10, 0.6),
             arrowprops=dict(arrowstyle='->', color='#e74c3c'),
             fontsize=9, color='#e74c3c',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax1.annotate('$\\gamma=0.99$: recompensas lejanas\naun valen mucho',
             xy=(20, 0.99**20), xytext=(10, 0.35),
             arrowprops=dict(arrowstyle='->', color='#2ecc71'),
             fontsize=9, color='#2ecc71',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Grafico 2: Ejemplo numerico de calculo de G_t
rewards_seq = [-1, -1, -1, 100]  # camino de 3 pasos hasta la meta
gamma_example = 0.9

# Calcular contribucion de cada recompensa
contributions = [gamma_example**k * r for k, r in enumerate(rewards_seq)]
G_t = sum(contributions)

bar_colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in contributions]
bar_labels = [f'$\\gamma^{k} \\times ({r})$\n$= {c:.2f}$'
              for k, (r, c) in enumerate(zip(rewards_seq, contributions))]

bars = ax2.bar(range(len(contributions)), contributions, color=bar_colors,
               edgecolor='black', linewidth=1.5, alpha=0.8)
ax2.set_xticks(range(len(contributions)))
ax2.set_xticklabels(bar_labels, fontsize=9)
ax2.set_ylabel('Contribucion al retorno $G_t$', fontsize=12)
ax2.set_title(f'Ejemplo: Calculo de $G_t$ con $\\gamma = {gamma_example}$',
              fontsize=13, fontweight='bold')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.axhline(y=G_t, color='blue', linewidth=2, linestyle='--',
            label=f'$G_t = {G_t:.2f}$')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# --- Tabla resumen numerica ---
print('Ejemplo numerico detallado:')
print(f'Secuencia de recompensas: {rewards_seq}')
print(f'Factor de descuento: gamma = {gamma_example}')
print()
print('Calculo paso a paso:')
formula_parts = []
for k, r in enumerate(rewards_seq):
    discount = gamma_example**k
    contribution = discount * r
    print(f'  k={k}: gamma^{k} x R_{k+1} = {discount:.4f} x ({r:+d}) = {contribution:+.4f}')
    formula_parts.append(f'{contribution:+.4f}')

print(f'\nG_t = {"+".join(formula_parts)} = {G_t:.4f}')
print(f'\nInterpretacion: a pesar de 3 pasos con recompensa -1,')
print(f'la recompensa de +100 en la meta hace que el retorno total sea positivo ({G_t:.2f}).')

## Ejemplo Numérico Paso a Paso

Para ver cómo funciona Q-Learning en la práctica, vamos a trabajar con un **GridWorld 3x3 simplificado** y calcular las actualizaciones manualmente.

El grid es:
- Inicio en **(0, 0)**
- Meta en **(2, 2)** con recompensa **+10**
- Obstáculo en **(1, 1)** con recompensa **-5**
- Todas las demás celdas: recompensa **-1**

Parámetros: $\alpha = 0.5$, $\gamma = 0.9$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- GridWorld 3x3 simplificado ---
grid_size = 3
start = (0, 0)
goal = (2, 2)
obstacle = (1, 1)

# Acciones: 0=arriba, 1=abajo, 2=izquierda, 3=derecha
action_names = ['arriba', 'abajo', 'izquierda', 'derecha']
n_actions = 4

# Desplazamientos
action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

# Recompensas
reward_map = np.full((grid_size, grid_size), -1.0)
reward_map[obstacle] = -5.0
reward_map[goal] = 10.0

# --- Visualizar el grid 3x3 ---
fig, ax = plt.subplots(figsize=(5, 5))
for i in range(grid_size):
    for j in range(grid_size):
        if (i, j) == start:
            color = '#90EE90'
            label = f'S\n({i},{j})'
        elif (i, j) == goal:
            color = '#FFD700'
            label = f'G\n({i},{j})\n+10'
        elif (i, j) == obstacle:
            color = '#FF6B6B'
            label = f'X\n({i},{j})\n-5'
        else:
            color = '#E8E8E8'
            label = f'({i},{j})\n-1'

        rect = plt.Rectangle((j, grid_size - 1 - i), 1, 1,
                             facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(j + 0.5, grid_size - 1 - i + 0.5, label,
                ha='center', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, grid_size)
ax.set_ylim(0, grid_size)
ax.set_aspect('equal')
ax.set_title('GridWorld 3x3 para ejemplo numerico', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

# --- Inicializar Q-table ---
Q = np.zeros((grid_size, grid_size, n_actions))

print('Q-table inicial (todos ceros):')
print('=' * 65)
print(f'{"Estado":<12} | {"arriba":>8} | {"abajo":>8} | {"izquierda":>10} | {"derecha":>8}')
print('-' * 65)
for i in range(grid_size):
    for j in range(grid_size):
        state_label = f'({i},{j})'
        if (i, j) == start:
            state_label += ' [S]'
        elif (i, j) == goal:
            state_label += ' [G]'
        elif (i, j) == obstacle:
            state_label += ' [X]'
        vals = Q[i, j]
        print(f'{state_label:<12} | {vals[0]:>8.2f} | {vals[1]:>8.2f} | {vals[2]:>10.2f} | {vals[3]:>8.2f}')
print('=' * 65)
print(f'\nDimensiones de la Q-table: {Q.shape} (filas x columnas x acciones)')
print(f'Total de entradas: {Q.size} valores Q a aprender')

In [ ]:
import numpy as np

# --- Q-Learning paso a paso manual ---
grid_size = 3
goal = (2, 2)
obstacle = (1, 1)
action_names = ['arriba', 'abajo', 'izquierda', 'derecha']
action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
n_actions = 4

reward_map = np.full((grid_size, grid_size), -1.0)
reward_map[obstacle] = -5.0
reward_map[goal] = 10.0

alpha = 0.5
gamma = 0.9

Q = np.zeros((grid_size, grid_size, n_actions))

def next_state(state, action_idx):
    """Calcula el siguiente estado dada una accion."""
    dr, dc = action_deltas[action_idx]
    new_r = max(0, min(grid_size - 1, state[0] + dr))
    new_c = max(0, min(grid_size - 1, state[1] + dc))
    return (new_r, new_c)

# Definimos 3 episodios con acciones predeterminadas para mostrar el calculo
episodes = [
    # Episodio 1: camino hacia abajo y derecha (corto)
    [(0,0), 1, (0,0), 3, (0,1), 1, (1,1), 3, (1,2), 1, (2,2)],
    # Episodio 2: camino por arriba para evitar obstaculo
    [(0,0), 3, (0,1), 3, (0,2), 1, (1,2), 1, (2,2)],
    # Episodio 3: repite camino optimo reforzando valores
    [(0,0), 3, (0,1), 3, (0,2), 1, (1,2), 1, (2,2)],
]

# Parsear episodios como secuencias de (estado, accion)
parsed_episodes = []
for ep in episodes:
    steps = []
    idx = 0
    while idx < len(ep) - 1:
        state = ep[idx]
        action = ep[idx + 1]
        idx += 2
        s_next = next_state(state, action)
        reward = reward_map[s_next]
        steps.append((state, action, reward, s_next))
    parsed_episodes.append(steps)

print('=== Q-LEARNING: ACTUALIZACIONES PASO A PASO ===')
print(f'Parametros: alpha = {alpha}, gamma = {gamma}')
print(f'Q-table inicial: todos ceros')
print()

for ep_num, steps in enumerate(parsed_episodes):
    print(f'{'='*70}')
    print(f'EPISODIO {ep_num + 1}')
    print(f'{'='*70}')

    for step_num, (s, a, r, s_next) in enumerate(steps):
        # Valores actuales
        q_old = Q[s[0], s[1], a]
        max_q_next = np.max(Q[s_next[0], s_next[1]])

        # TD error
        td_target = r + gamma * max_q_next
        td_error = td_target - q_old

        # Actualizar
        q_new = q_old + alpha * td_error
        Q[s[0], s[1], a] = q_new

        # Mostrar calculo detallado
        print(f'\n  Paso {step_num + 1}: s={s}, a={action_names[a]}, R={r:+.1f}, s\' = {s_next}')
        print(f'    Q_viejo({s}, {action_names[a]}) = {q_old:.4f}')
        print(f'    max_a\' Q({s_next}, a\') = {max_q_next:.4f}')
        print(f'    TD target = R + gamma * max Q = {r:.1f} + {gamma} * {max_q_next:.4f} = {td_target:.4f}')
        print(f'    TD error  = {td_target:.4f} - {q_old:.4f} = {td_error:.4f}')
        print(f'    Q_nuevo   = {q_old:.4f} + {alpha} * {td_error:.4f} = {q_new:.4f}')

        if s_next == goal:
            print(f'\n    >>> Llego a la META! Fin del episodio {ep_num + 1}.')

    # Mostrar Q-table despues del episodio
    print(f'\n  Q-table despues del Episodio {ep_num + 1}:')
    print(f'  {"Estado":<10} | {"arriba":>8} | {"abajo":>8} | {"izq":>8} | {"der":>8}')
    print(f'  {"-"*55}')
    for i in range(grid_size):
        for j in range(grid_size):
            vals = Q[i, j]
            marker = ''
            if any(v != 0 for v in vals):
                marker = ' *'  # marcar celdas que cambiaron
            print(f'  ({i},{j}){marker:<5} | {vals[0]:>8.4f} | {vals[1]:>8.4f} | {vals[2]:>8.4f} | {vals[3]:>8.4f}')
    print()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Entrenamiento completo y visualizacion de Q-values ---
grid_size = 3
goal = (2, 2)
obstacle = (1, 1)
start = (0, 0)
action_names = ['arriba', 'abajo', 'izquierda', 'derecha']
action_arrows = {'arriba': '\u2191', 'abajo': '\u2193', 'izquierda': '\u2190', 'derecha': '\u2192'}
action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
n_actions = 4

reward_map = np.full((grid_size, grid_size), -1.0)
reward_map[obstacle] = -5.0
reward_map[goal] = 10.0

alpha = 0.1
gamma = 0.9
epsilon = 0.3

def get_next_state(state, action_idx):
    dr, dc = action_deltas[action_idx]
    new_r = max(0, min(grid_size - 1, state[0] + dr))
    new_c = max(0, min(grid_size - 1, state[1] + dc))
    return (new_r, new_c)

def run_q_learning(n_episodes, alpha, gamma, epsilon):
    """Ejecuta Q-Learning y guarda Q-tables intermedias."""
    Q = np.zeros((grid_size, grid_size, n_actions))
    snapshots = {}
    snapshot_points = [0, 10, 50, 200, 500]

    for episode in range(n_episodes + 1):
        if episode in snapshot_points:
            snapshots[episode] = Q.copy()

        state = start
        steps = 0
        max_steps = 100

        while state != goal and steps < max_steps:
            # Epsilon-greedy
            if np.random.random() < epsilon:
                action = np.random.randint(n_actions)
            else:
                action = np.argmax(Q[state[0], state[1]])

            s_next = get_next_state(state, action)
            reward = reward_map[s_next]

            # Q-Learning update
            q_old = Q[state[0], state[1], action]
            max_q_next = np.max(Q[s_next[0], s_next[1]])
            Q[state[0], state[1], action] = q_old + alpha * (reward + gamma * max_q_next - q_old)

            state = s_next
            steps += 1

    return snapshots

np.random.seed(42)
snapshots = run_q_learning(500, alpha, gamma, epsilon)

# --- Heatmaps de max Q-value en cada estado ---
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
snapshot_points = [0, 10, 50, 200, 500]

for ax, ep in zip(axes, snapshot_points):
    Q_snap = snapshots[ep]
    # Valor maximo Q para cada estado
    V = np.max(Q_snap, axis=2)

    im = ax.imshow(V, cmap='YlOrRd', interpolation='nearest', vmin=-5, vmax=10)

    # Anotar valores y flechas de politica
    for i in range(grid_size):
        for j in range(grid_size):
            val = V[i, j]
            best_a = np.argmax(Q_snap[i, j])
            arrow = action_arrows[action_names[best_a]]

            if (i, j) == goal:
                text = f'META\n{val:.1f}'
            elif (i, j) == obstacle:
                text = f'X\n{val:.1f}'
            else:
                text = f'{arrow}\n{val:.1f}'

            color = 'white' if val > 5 else 'black'
            ax.text(j, i, text, ha='center', va='center',
                    fontsize=10, fontweight='bold', color=color)

    ax.set_title(f'Episodio {ep}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.set_xticklabels(['Col 0', 'Col 1', 'Col 2'], fontsize=8)
    ax.set_yticklabels(['Fila 0', 'Fila 1', 'Fila 2'], fontsize=8)

fig.colorbar(im, ax=axes[-1], shrink=0.8, label='max Q(s,a)')
plt.suptitle('Evolucion de los valores Q durante el entrenamiento\n'
             '(los valores se propagan desde la meta hacia atras)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observaciones:')
print('  - Episodio 0: Todos los Q-values son 0 (el agente no sabe nada).')
print('  - Episodio 10: Los estados cercanos a la meta empiezan a tener valores positivos.')
print('  - Episodio 50: Los valores se propagan mas lejos desde la meta.')
print('  - Episodio 200-500: Los valores convergen. Las flechas muestran la politica aprendida.')
print('  - Notar como el obstaculo (X) tiene valores bajos, y la meta (META) los mas altos.')
print('  - Este fenomeno de propagacion hacia atras es la esencia de las ecuaciones de Bellman.')

---

## Implementación con Código

Ahora implementemos los algoritmos Q-Learning y SARSA desde cero, primero con funciones individuales y luego como una clase reutilizable.

### Funciones

In [ ]:
import numpy as np

class GridWorld:
    """
    Entorno GridWorld configurable para Reinforcement Learning.
    
    Permite definir grids de cualquier tamaño con obstáculos,
    trampas, y recompensas personalizadas.
    
    Parámetros
    ----------
    filas : int
        Número de filas del grid.
    columnas : int
        Número de columnas del grid.
    inicio : tuple
        Posición inicial (fila, columna).
    meta : tuple
        Posición de la meta.
    obstaculos : list de tuples
        Posiciones de los obstáculos.
    recompensa_paso : float
        Recompensa por cada paso normal.
    recompensa_obstaculo : float
        Recompensa por pisar un obstáculo.
    recompensa_meta : float
        Recompensa por llegar a la meta.
    """
    
    # Acciones: 0=arriba, 1=abajo, 2=izquierda, 3=derecha
    ACCIONES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    NOMBRES_ACCIONES = {0: 'arriba', 1: 'abajo', 2: 'izquierda', 3: 'derecha'}
    FLECHAS = {0: '↑', 1: '↓', 2: '←', 3: '→'}
    N_ACCIONES = 4
    
    def __init__(self, filas=4, columnas=4, inicio=(0,0), meta=(3,3),
                 obstaculos=None, recompensa_paso=-1, recompensa_obstaculo=-10,
                 recompensa_meta=100):
        self.filas = filas
        self.columnas = columnas
        self.inicio = inicio
        self.meta = meta
        self.obstaculos = obstaculos or []
        self.recompensa_paso = recompensa_paso
        self.recompensa_obstaculo = recompensa_obstaculo
        self.recompensa_meta = recompensa_meta
        
        self.n_estados = filas * columnas
        self.estado_actual = inicio
        
        # Mapa de recompensas
        self.recompensas = np.full((filas, columnas), recompensa_paso)
        for obs in self.obstaculos:
            self.recompensas[obs] = recompensa_obstaculo
        self.recompensas[meta] = recompensa_meta
    
    def reset(self):
        """Reinicia el entorno al estado inicial."""
        self.estado_actual = self.inicio
        return self.estado_actual
    
    def step(self, accion):
        """
        Ejecuta una acción y retorna (nuevo_estado, recompensa, terminado).
        """
        dr, dc = self.ACCIONES[accion]
        nueva_fila = max(0, min(self.filas - 1, self.estado_actual[0] + dr))
        nueva_col = max(0, min(self.columnas - 1, self.estado_actual[1] + dc))
        
        nuevo_estado = (nueva_fila, nueva_col)
        recompensa = self.recompensas[nuevo_estado]
        terminado = (nuevo_estado == self.meta)
        
        self.estado_actual = nuevo_estado
        return nuevo_estado, recompensa, terminado
    
    def estados(self):
        """Retorna lista de todos los estados."""
        return [(i, j) for i in range(self.filas) for j in range(self.columnas)]


# Prueba
env = GridWorld(filas=4, columnas=4, inicio=(0,0), meta=(3,3), 
                obstaculos=[(1,1)], recompensa_meta=100)
s = env.reset()
print(f"Estado inicial: {s}")
s2, r, done = env.step(1)  # abajo
print(f"Acción: abajo → estado={s2}, recompensa={r}, terminado={done}")
s3, r, done = env.step(3)  # derecha
print(f"Acción: derecha → estado={s3}, recompensa={r}, terminado={done}")
print(f"\nEntorno GridWorld definido correctamente ✓")

In [ ]:
def epsilon_greedy(Q, estado, epsilon, n_acciones=4):
    """
    Selecciona una acción usando la estrategia epsilon-greedy.
    
    Con probabilidad epsilon: acción aleatoria (exploración).
    Con probabilidad 1-epsilon: mejor acción conocida (explotación).
    """
    if np.random.random() < epsilon:
        return np.random.randint(n_acciones)
    else:
        return np.argmax(Q[estado[0], estado[1]])


def q_learning(env, n_episodios=1000, alpha=0.1, gamma=0.99, 
               epsilon_inicio=1.0, epsilon_fin=0.01, epsilon_decay=0.995,
               max_pasos=200):
    """
    Algoritmo Q-Learning (off-policy TD control).
    
    Q(s,a) ← Q(s,a) + α[R + γ·max_a' Q(s',a') - Q(s,a)]
    
    Parámetros
    ----------
    env : GridWorld
        Entorno.
    n_episodios : int
        Número de episodios de entrenamiento.
    alpha : float
        Tasa de aprendizaje.
    gamma : float
        Factor de descuento.
    epsilon_inicio, epsilon_fin, epsilon_decay : float
        Parámetros de exploración.
    max_pasos : int
        Máximo de pasos por episodio.
    
    Retorna
    -------
    Q : array
        Tabla Q aprendida.
    recompensas_por_episodio : list
        Recompensa total de cada episodio.
    pasos_por_episodio : list
        Pasos por episodio.
    """
    Q = np.zeros((env.filas, env.columnas, env.N_ACCIONES))
    recompensas_por_episodio = []
    pasos_por_episodio = []
    epsilon = epsilon_inicio
    
    for episodio in range(n_episodios):
        estado = env.reset()
        recompensa_total = 0
        
        for paso in range(max_pasos):
            # Seleccionar acción (epsilon-greedy)
            accion = epsilon_greedy(Q, estado, epsilon)
            
            # Ejecutar acción
            nuevo_estado, recompensa, terminado = env.step(accion)
            recompensa_total += recompensa
            
            # Actualización Q-Learning (usa max)
            q_actual = Q[estado[0], estado[1], accion]
            max_q_siguiente = np.max(Q[nuevo_estado[0], nuevo_estado[1]])
            td_target = recompensa + gamma * max_q_siguiente * (1 - terminado)
            Q[estado[0], estado[1], accion] += alpha * (td_target - q_actual)
            
            estado = nuevo_estado
            
            if terminado:
                break
        
        recompensas_por_episodio.append(recompensa_total)
        pasos_por_episodio.append(paso + 1)
        epsilon = max(epsilon_fin, epsilon * epsilon_decay)
    
    return Q, recompensas_por_episodio, pasos_por_episodio


def sarsa(env, n_episodios=1000, alpha=0.1, gamma=0.99,
          epsilon_inicio=1.0, epsilon_fin=0.01, epsilon_decay=0.995,
          max_pasos=200):
    """
    Algoritmo SARSA (on-policy TD control).
    
    Q(s,a) ← Q(s,a) + α[R + γ·Q(s',a') - Q(s,a)]
    
    La diferencia con Q-Learning: usa Q(s',a') en vez de max_a' Q(s',a').
    """
    Q = np.zeros((env.filas, env.columnas, env.N_ACCIONES))
    recompensas_por_episodio = []
    pasos_por_episodio = []
    epsilon = epsilon_inicio
    
    for episodio in range(n_episodios):
        estado = env.reset()
        accion = epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        
        for paso in range(max_pasos):
            nuevo_estado, recompensa, terminado = env.step(accion)
            recompensa_total += recompensa
            
            # Seleccionar SIGUIENTE acción (la que realmente tomará)
            nueva_accion = epsilon_greedy(Q, nuevo_estado, epsilon)
            
            # Actualización SARSA (usa Q de la acción real)
            q_actual = Q[estado[0], estado[1], accion]
            q_siguiente = Q[nuevo_estado[0], nuevo_estado[1], nueva_accion]
            td_target = recompensa + gamma * q_siguiente * (1 - terminado)
            Q[estado[0], estado[1], accion] += alpha * (td_target - q_actual)
            
            estado = nuevo_estado
            accion = nueva_accion
            
            if terminado:
                break
        
        recompensas_por_episodio.append(recompensa_total)
        pasos_por_episodio.append(paso + 1)
        epsilon = max(epsilon_fin, epsilon * epsilon_decay)
    
    return Q, recompensas_por_episodio, pasos_por_episodio


print('Funciones q_learning() y sarsa() definidas correctamente ✓')

In [ ]:
import matplotlib.pyplot as plt

def visualizar_politica(Q, env, titulo='Política aprendida', ax=None):
    """Visualiza la política como flechas en el grid."""
    show = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(env.columnas*1.5, env.filas*1.5))
        show = True
    
    for i in range(env.filas):
        for j in range(env.columnas):
            if (i,j) == env.inicio:
                color = '#90EE90'
            elif (i,j) == env.meta:
                color = '#FFD700'
            elif (i,j) in env.obstaculos:
                color = '#FF6B6B'
            else:
                color = '#E8E8E8'
            
            rect = plt.Rectangle((j, env.filas-1-i), 1, 1, facecolor=color, 
                                edgecolor='black', linewidth=2)
            ax.add_patch(rect)
            
            if (i,j) == env.meta:
                ax.text(j+0.5, env.filas-1-i+0.5, 'META', ha='center', va='center',
                       fontsize=10, fontweight='bold')
            elif (i,j) in env.obstaculos:
                ax.text(j+0.5, env.filas-1-i+0.5, 'X', ha='center', va='center',
                       fontsize=14, fontweight='bold', color='white')
            else:
                best_a = np.argmax(Q[i, j])
                flecha = env.FLECHAS[best_a]
                ax.text(j+0.5, env.filas-1-i+0.5, flecha, ha='center', va='center',
                       fontsize=20, fontweight='bold')
    
    ax.set_xlim(0, env.columnas); ax.set_ylim(0, env.filas)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    if show:
        plt.tight_layout(); plt.show()


def visualizar_valores(Q, env, titulo='Valores V(s) = max Q(s,a)', ax=None):
    """Visualiza los valores de estado como heatmap."""
    show = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(env.columnas*1.5, env.filas*1.5))
        show = True
    
    V = np.max(Q, axis=2)
    im = ax.imshow(V, cmap='YlOrRd', interpolation='nearest')
    
    for i in range(env.filas):
        for j in range(env.columnas):
            color = 'white' if V[i,j] > (V.max()+V.min())/2 else 'black'
            ax.text(j, i, f'{V[i,j]:.1f}', ha='center', va='center',
                   fontsize=10, fontweight='bold', color=color)
    
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)
    if show:
        plt.tight_layout(); plt.show()


def plot_aprendizaje(recompensas, pasos=None, titulo='Curva de aprendizaje', ventana=50):
    """Grafica la curva de aprendizaje."""
    fig, axes = plt.subplots(1, 2 if pasos else 1, figsize=(14 if pasos else 8, 5))
    if not pasos:
        axes = [axes]
    
    # Recompensas
    ax = axes[0] if pasos else axes[0]
    ax.plot(recompensas, alpha=0.3, color='blue')
    if len(recompensas) >= ventana:
        media_movil = np.convolve(recompensas, np.ones(ventana)/ventana, mode='valid')
        ax.plot(range(ventana-1, len(recompensas)), media_movil, color='red', linewidth=2,
               label=f'Media móvil ({ventana} eps)')
    ax.set_xlabel('Episodio', fontsize=12)
    ax.set_ylabel('Recompensa total', fontsize=12)
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    if pasos:
        ax = axes[1]
        ax.plot(pasos, alpha=0.3, color='green')
        if len(pasos) >= ventana:
            media_pasos = np.convolve(pasos, np.ones(ventana)/ventana, mode='valid')
            ax.plot(range(ventana-1, len(pasos)), media_pasos, color='darkgreen', linewidth=2,
                   label=f'Media móvil ({ventana} eps)')
        ax.set_xlabel('Episodio', fontsize=12)
        ax.set_ylabel('Pasos por episodio', fontsize=12)
        ax.set_title('Pasos para llegar a la meta', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout(); plt.show()


print('Funciones de visualización definidas ✓')

### Clase

In [ ]:
class AgenteRL:
    """
    Agente de Reinforcement Learning con Q-Learning y SARSA.
    
    Parámetros
    ----------
    env : GridWorld
        Entorno donde opera el agente.
    algoritmo : str, 'q_learning' o 'sarsa'
        Algoritmo de aprendizaje.
    alpha : float
        Tasa de aprendizaje.
    gamma : float
        Factor de descuento.
    epsilon_inicio : float
        Epsilon inicial para exploración.
    epsilon_fin : float
        Epsilon mínimo.
    epsilon_decay : float
        Factor de decaimiento de epsilon.
    """
    
    def __init__(self, env, algoritmo='q_learning', alpha=0.1, gamma=0.99,
                 epsilon_inicio=1.0, epsilon_fin=0.01, epsilon_decay=0.995):
        self.env = env
        self.algoritmo = algoritmo
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon_inicio = epsilon_inicio
        self.epsilon_fin = epsilon_fin
        self.epsilon_decay = epsilon_decay
        
        self.Q = np.zeros((env.filas, env.columnas, env.N_ACCIONES))
        self.epsilon = epsilon_inicio
        self.recompensas_historial = []
        self.pasos_historial = []
    
    def _seleccionar_accion(self, estado):
        """Selección epsilon-greedy."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.env.N_ACCIONES)
        return np.argmax(self.Q[estado[0], estado[1]])
    
    def entrenar(self, n_episodios=1000, max_pasos=200, verbose=True):
        """
        Entrena al agente durante n_episodios.
        """
        self.Q = np.zeros((self.env.filas, self.env.columnas, self.env.N_ACCIONES))
        self.epsilon = self.epsilon_inicio
        self.recompensas_historial = []
        self.pasos_historial = []
        
        for ep in range(n_episodios):
            estado = self.env.reset()
            recompensa_total = 0
            
            if self.algoritmo == 'sarsa':
                accion = self._seleccionar_accion(estado)
            
            for paso in range(max_pasos):
                if self.algoritmo == 'q_learning':
                    accion = self._seleccionar_accion(estado)
                
                nuevo_estado, recompensa, terminado = self.env.step(accion)
                recompensa_total += recompensa
                
                if self.algoritmo == 'q_learning':
                    # Q-Learning: usa max
                    objetivo = recompensa + self.gamma * np.max(self.Q[nuevo_estado[0], nuevo_estado[1]]) * (1 - terminado)
                    self.Q[estado[0], estado[1], accion] += self.alpha * (objetivo - self.Q[estado[0], estado[1], accion])
                else:
                    # SARSA: usa acción real
                    nueva_accion = self._seleccionar_accion(nuevo_estado)
                    objetivo = recompensa + self.gamma * self.Q[nuevo_estado[0], nuevo_estado[1], nueva_accion] * (1 - terminado)
                    self.Q[estado[0], estado[1], accion] += self.alpha * (objetivo - self.Q[estado[0], estado[1], accion])
                    accion = nueva_accion
                
                estado = nuevo_estado
                if terminado:
                    break
            
            self.recompensas_historial.append(recompensa_total)
            self.pasos_historial.append(paso + 1)
            self.epsilon = max(self.epsilon_fin, self.epsilon * self.epsilon_decay)
        
        if verbose:
            ult = self.recompensas_historial[-100:]
            print(f'{self.algoritmo.upper()} - {n_episodios} episodios')
            print(f'  Recompensa media (últimos 100): {np.mean(ult):.1f}')
            print(f'  Pasos medio (últimos 100): {np.mean(self.pasos_historial[-100:]):.1f}')
    
    def ejecutar_episodio(self, render=True):
        """Ejecuta un episodio con la política aprendida (sin exploración)."""
        estado = self.env.reset()
        camino = [estado]
        recompensa_total = 0
        
        for _ in range(200):
            accion = np.argmax(self.Q[estado[0], estado[1]])
            estado, r, done = self.env.step(accion)
            camino.append(estado)
            recompensa_total += r
            if done:
                break
        
        if render:
            print(f'Camino: {" → ".join([str(s) for s in camino])}')
            print(f'Pasos: {len(camino)-1}, Recompensa total: {recompensa_total:.1f}')
        
        return camino, recompensa_total
    
    def plot_aprendizaje(self, ventana=50):
        """Grafica la curva de aprendizaje."""
        plot_aprendizaje(self.recompensas_historial, self.pasos_historial,
                        f'Aprendizaje - {self.algoritmo.upper()}', ventana)
    
    def plot_politica(self, titulo=None):
        """Visualiza la política aprendida."""
        if titulo is None:
            titulo = f'Política - {self.algoritmo.upper()}'
        visualizar_politica(self.Q, self.env, titulo)
    
    def plot_valores(self, titulo=None):
        """Visualiza los valores de estado."""
        if titulo is None:
            titulo = f'Valores V(s) - {self.algoritmo.upper()}'
        visualizar_valores(self.Q, self.env, titulo)


print('Clase AgenteRL definida correctamente ✓')

---

## Experimentos

### Ejemplo 1: GridWorld 4x4 básico

In [ ]:
np.random.seed(42)

# Entorno GridWorld 4x4
env1 = GridWorld(filas=4, columnas=4, inicio=(0,0), meta=(3,3),
                 obstaculos=[(1,1)], recompensa_meta=100)

# Entrenar con Q-Learning
agente1 = AgenteRL(env1, algoritmo='q_learning', alpha=0.1, gamma=0.99)
agente1.entrenar(n_episodios=2000)

# Visualizar resultados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
visualizar_politica(agente1.Q, env1, 'Política Q-Learning', ax=axes[0])
visualizar_valores(agente1.Q, env1, 'Valores V(s)', ax=axes[1])
plt.tight_layout()
plt.show()

# Ejecutar episodio óptimo
print('\nEpisodio con política aprendida:')
agente1.ejecutar_episodio()

# Curva de aprendizaje
agente1.plot_aprendizaje()

### Ejemplo 2: Q-Learning vs SARSA

In [ ]:
np.random.seed(42)

env2 = GridWorld(filas=4, columnas=4, inicio=(0,0), meta=(3,3),
                 obstaculos=[(1,1), (2,2)], recompensa_meta=100)

# Q-Learning
agente_ql = AgenteRL(env2, algoritmo='q_learning', alpha=0.1, gamma=0.99)
agente_ql.entrenar(n_episodios=3000)

# SARSA
np.random.seed(42)
agente_sarsa = AgenteRL(env2, algoritmo='sarsa', alpha=0.1, gamma=0.99)
agente_sarsa.entrenar(n_episodios=3000)

# Comparar políticas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
visualizar_politica(agente_ql.Q, env2, 'Q-Learning', ax=axes[0])
visualizar_politica(agente_sarsa.Q, env2, 'SARSA', ax=axes[1])
plt.suptitle('Q-Learning vs SARSA (2 obstáculos)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Comparar curvas
fig, ax = plt.subplots(figsize=(10, 5))
v = 100
r_ql = np.convolve(agente_ql.recompensas_historial, np.ones(v)/v, mode='valid')
r_sa = np.convolve(agente_sarsa.recompensas_historial, np.ones(v)/v, mode='valid')
ax.plot(r_ql, label='Q-Learning', linewidth=2, color='#e74c3c')
ax.plot(r_sa, label='SARSA', linewidth=2, color='#3498db')
ax.set_xlabel('Episodio', fontsize=12)
ax.set_ylabel('Recompensa (media móvil)', fontsize=12)
ax.set_title('Q-Learning vs SARSA: Convergencia', fontsize=13, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('Camino Q-Learning:'); agente_ql.ejecutar_episodio()
print('\nCamino SARSA:'); agente_sarsa.ejecutar_episodio()

### Ejemplo 3: Cliff Walking (el acantilado)

Este es el ejemplo clásico donde se ve la diferencia entre Q-Learning y SARSA.

El agente debe ir de la esquina inferior izquierda a la inferior derecha, pero el borde inferior es un **acantilado** que da una penalización enorme.

In [ ]:
class CliffWalking(GridWorld):
    """Entorno Cliff Walking: acantilado en el borde inferior."""
    
    def __init__(self):
        # Grid 4x12, inicio (3,0), meta (3,11)
        # El acantilado son las celdas (3,1) a (3,10)
        cliff = [(3, j) for j in range(1, 11)]
        super().__init__(filas=4, columnas=12, inicio=(3,0), meta=(3,11),
                        obstaculos=cliff, recompensa_paso=-1,
                        recompensa_obstaculo=-100, recompensa_meta=0)
    
    def step(self, accion):
        """Si cae al acantilado, vuelve al inicio."""
        nuevo_estado, recompensa, terminado = super().step(accion)
        
        if nuevo_estado in self.obstaculos:
            self.estado_actual = self.inicio
            return self.inicio, recompensa, False
        
        return nuevo_estado, recompensa, terminado


np.random.seed(42)
cliff_env = CliffWalking()

# Visualizar el entorno
fig, ax = plt.subplots(figsize=(16, 4))
for i in range(4):
    for j in range(12):
        if (i,j) == cliff_env.inicio:
            color = '#90EE90'; label = 'S'
        elif (i,j) == cliff_env.meta:
            color = '#FFD700'; label = 'G'
        elif (i,j) in cliff_env.obstaculos:
            color = '#8B0000'; label = '☠'
        else:
            color = '#E8E8E8'; label = ''
        rect = plt.Rectangle((j, 3-i), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        ax.text(j+0.5, 3-i+0.5, label, ha='center', va='center', fontsize=12, fontweight='bold',
               color='white' if color == '#8B0000' else 'black')
ax.set_xlim(0,12); ax.set_ylim(0,4); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Cliff Walking: S=inicio, G=meta, ☠=acantilado (-100)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Q-Learning (off-policy → camino óptimo pero arriesgado)
agente_cliff_ql = AgenteRL(cliff_env, algoritmo='q_learning', alpha=0.5, gamma=0.99,
                            epsilon_inicio=0.1, epsilon_fin=0.1, epsilon_decay=1.0)
agente_cliff_ql.entrenar(n_episodios=500)

# SARSA (on-policy → camino seguro)
np.random.seed(42)
cliff_env2 = CliffWalking()
agente_cliff_sa = AgenteRL(cliff_env2, algoritmo='sarsa', alpha=0.5, gamma=0.99,
                            epsilon_inicio=0.1, epsilon_fin=0.1, epsilon_decay=1.0)
agente_cliff_sa.entrenar(n_episodios=500)

# Comparar caminos
fig, axes = plt.subplots(1, 2, figsize=(20, 4))
for ax, agente, titulo in [(axes[0], agente_cliff_ql, 'Q-Learning (camino óptimo/arriesgado)'),
                            (axes[1], agente_cliff_sa, 'SARSA (camino seguro)')]:
    for i in range(4):
        for j in range(12):
            if (i,j) == cliff_env.inicio:
                color = '#90EE90'
            elif (i,j) == cliff_env.meta:
                color = '#FFD700'
            elif (i,j) in cliff_env.obstaculos:
                color = '#8B0000'
            else:
                color = '#E8E8E8'
            rect = plt.Rectangle((j, 3-i), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            
            if (i,j) not in cliff_env.obstaculos and (i,j) != cliff_env.meta:
                best_a = np.argmax(agente.Q[i,j])
                ax.text(j+0.5, 3-i+0.5, cliff_env.FLECHAS[best_a], ha='center', va='center',
                       fontsize=14, fontweight='bold')
    ax.set_xlim(0,12); ax.set_ylim(0,4); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
plt.suptitle('Cliff Walking: Q-Learning vs SARSA', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

# Curvas
fig, ax = plt.subplots(figsize=(10, 5))
v = 20
ax.plot(np.convolve(agente_cliff_ql.recompensas_historial, np.ones(v)/v, mode='valid'),
        label='Q-Learning', linewidth=2, color='#e74c3c')
ax.plot(np.convolve(agente_cliff_sa.recompensas_historial, np.ones(v)/v, mode='valid'),
        label='SARSA', linewidth=2, color='#3498db')
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa')
ax.set_title('Cliff Walking: Q-Learning vs SARSA', fontsize=13, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('Q-Learning aprende el camino ÓPTIMO (pegado al acantilado) → más recompensa pero arriesgado')
print('SARSA aprende un camino SEGURO (lejos del acantilado) → menos recompensa pero más estable')

### Ejemplo 4: Frozen Lake (lago congelado)

Un entorno estocástico: el agente camina sobre un lago congelado, pero el hielo es resbaladizo. Con cierta probabilidad, se mueve en una dirección diferente a la elegida.

In [ ]:
class FrozenLake(GridWorld):
    """Frozen Lake: entorno estocástico donde el agente puede resbalar."""
    
    def __init__(self, prob_resbalar=0.2):
        # H = hielo seguro, F = agujero (hole), S = start, G = goal
        agujeros = [(1,1), (1,3), (2,3), (3,0)]
        super().__init__(filas=4, columnas=4, inicio=(0,0), meta=(3,3),
                        obstaculos=agujeros, recompensa_paso=-0.01,
                        recompensa_obstaculo=-1, recompensa_meta=1)
        self.prob_resbalar = prob_resbalar
    
    def step(self, accion):
        """Con cierta probabilidad, la acción se desvía."""
        if np.random.random() < self.prob_resbalar:
            accion = np.random.randint(self.N_ACCIONES)
        
        nuevo_estado, recompensa, terminado = super().step(accion)
        
        if nuevo_estado in self.obstaculos:
            return nuevo_estado, recompensa, True
        
        return nuevo_estado, recompensa, terminado


# Entrenar en entorno estocástico
np.random.seed(42)
lake_env = FrozenLake(prob_resbalar=0.2)

# Visualizar
fig, ax = plt.subplots(figsize=(7, 7))
mapa = {(0,0): ('S', '#90EE90'), (3,3): ('G', '#FFD700'),
        (1,1): ('☠', '#4169E1'), (1,3): ('☠', '#4169E1'),
        (2,3): ('☠', '#4169E1'), (3,0): ('☠', '#4169E1')}
for i in range(4):
    for j in range(4):
        label, color = mapa.get((i,j), ('❄', '#ADD8E6'))
        rect = plt.Rectangle((j, 3-i), 1, 1, facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(j+0.5, 3-i+0.5, label, ha='center', va='center', fontsize=16)
ax.set_xlim(0,4); ax.set_ylim(0,4); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Frozen Lake (20% prob. de resbalar)\nS=inicio, G=meta, ☠=agujero, ❄=hielo',
            fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Comparar determinista vs estocástico
resultados = {}
for prob in [0.0, 0.1, 0.3]:
    np.random.seed(42)
    env_prob = FrozenLake(prob_resbalar=prob)
    agente = AgenteRL(env_prob, algoritmo='q_learning', alpha=0.1, gamma=0.99)
    agente.entrenar(n_episodios=5000, verbose=False)
    resultados[prob] = agente

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (prob, agente) in zip(axes, resultados.items()):
    visualizar_politica(agente.Q, agente.env, f'Prob. resbalar = {prob:.0%}', ax=ax)
plt.suptitle('Efecto de la estocasticidad en la política', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('A mayor probabilidad de resbalar, el agente necesita políticas más robustas.')
print('El agente se adapta a la incertidumbre del entorno.')

### Ejemplo 5: Multi-Armed Bandit (máquinas tragamonedas)

El problema más simple de RL: elegir entre K máquinas tragamonedas, cada una con una probabilidad diferente de dar premio. No hay estados, solo acciones y recompensas.

In [ ]:
class MultiArmedBandit:
    """K máquinas tragamonedas con probabilidades ocultas."""
    
    def __init__(self, probabilidades):
        self.probabilidades = np.array(probabilidades)
        self.k = len(probabilidades)
        self.mejor_accion = np.argmax(probabilidades)
    
    def pull(self, accion):
        """Tira la palanca de la máquina 'accion'."""
        return 1.0 if np.random.random() < self.probabilidades[accion] else 0.0


def epsilon_greedy_bandit(Q, N, epsilon):
    """Selección epsilon-greedy para bandits."""
    if np.random.random() < epsilon:
        return np.random.randint(len(Q))
    return np.argmax(Q)


def resolver_bandit(bandit, n_pasos=1000, epsilon=0.1):
    """Resuelve un bandit con epsilon-greedy."""
    Q = np.zeros(bandit.k)  # Valor estimado de cada máquina
    N = np.zeros(bandit.k)  # Veces que se ha jugado cada máquina
    recompensas = []
    acciones_optimas = []
    
    for t in range(n_pasos):
        accion = epsilon_greedy_bandit(Q, N, epsilon)
        recompensa = bandit.pull(accion)
        
        N[accion] += 1
        Q[accion] += (recompensa - Q[accion]) / N[accion]
        
        recompensas.append(recompensa)
        acciones_optimas.append(accion == bandit.mejor_accion)
    
    return Q, recompensas, acciones_optimas


# Crear bandit con 5 máquinas
np.random.seed(42)
probs = [0.1, 0.3, 0.7, 0.2, 0.5]
bandit = MultiArmedBandit(probs)

print('Multi-Armed Bandit con 5 máquinas:')
for i, p in enumerate(probs):
    marca = ' ← MEJOR' if i == bandit.mejor_accion else ''
    print(f'  Máquina {i}: P(premio) = {p:.1f}{marca}')

# Comparar diferentes valores de epsilon
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epsilons = [0.01, 0.1, 0.3]
colores = ['#e74c3c', '#3498db', '#2ecc71']

for eps, color in zip(epsilons, colores):
    # Promediar 200 ejecuciones
    todas_recompensas = []
    todas_optimas = []
    for _ in range(200):
        _, rew, opt = resolver_bandit(bandit, n_pasos=1000, epsilon=eps)
        todas_recompensas.append(rew)
        todas_optimas.append(opt)
    
    media_rew = np.mean(todas_recompensas, axis=0)
    media_opt = np.mean(todas_optimas, axis=0)
    
    # Media acumulada
    axes[0].plot(np.cumsum(media_rew) / (np.arange(1000)+1), label=f'ε={eps}', 
                color=color, linewidth=2)
    axes[1].plot(np.convolve(media_opt, np.ones(50)/50, mode='valid'),
                label=f'ε={eps}', color=color, linewidth=2)

axes[0].axhline(y=max(probs), color='black', linestyle='--', alpha=0.5, label='Óptimo')
axes[0].set_xlabel('Paso'); axes[0].set_ylabel('Recompensa media acumulada')
axes[0].set_title('Recompensa acumulada', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Paso'); axes[1].set_ylabel('% acción óptima')
axes[1].set_title('% veces que elige la mejor máquina', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

# Valores Q aprendidos
Q_final, _, _ = resolver_bandit(bandit, n_pasos=5000, epsilon=0.1)
x_pos = np.arange(bandit.k)
axes[2].bar(x_pos - 0.15, probs, 0.3, label='P(premio) real', color='#3498db', alpha=0.7)
axes[2].bar(x_pos + 0.15, Q_final, 0.3, label='Q estimado', color='#e74c3c', alpha=0.7)
axes[2].set_xlabel('Máquina'); axes[2].set_ylabel('Valor')
axes[2].set_title('Valores reales vs estimados', fontsize=13, fontweight='bold')
axes[2].set_xticks(x_pos); axes[2].legend(fontsize=10); axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Multi-Armed Bandit: Exploración vs Explotación', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Observaciones:')
print('• ε=0.01: converge lento pero llega casi al óptimo (poca exploración)')
print('• ε=0.1: buen balance entre exploración y explotación')
print('• ε=0.3: explora demasiado, nunca explota completamente')

### Ejemplo 6: Laberinto 8x8 con múltiples obstáculos

Un entorno más complejo para poner a prueba el algoritmo.

In [ ]:
np.random.seed(42)

# Laberinto 8x8 con paredes
paredes = [
    (1,1), (1,2), (1,5), (1,6),
    (2,3), (2,4),
    (3,1), (3,6),
    (4,3), (4,4), (4,5),
    (5,1), (5,2),
    (6,4), (6,5), (6,6),
]

env_maze = GridWorld(filas=8, columnas=8, inicio=(0,0), meta=(7,7),
                     obstaculos=paredes, recompensa_paso=-1,
                     recompensa_obstaculo=-20, recompensa_meta=200)

# Entrenar
agente_maze = AgenteRL(env_maze, algoritmo='q_learning', alpha=0.1, gamma=0.99,
                        epsilon_inicio=1.0, epsilon_fin=0.01, epsilon_decay=0.999)
agente_maze.entrenar(n_episodios=10000)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
visualizar_politica(agente_maze.Q, env_maze, 'Política del laberinto 8x8', ax=axes[0])
visualizar_valores(agente_maze.Q, env_maze, 'Valores V(s) del laberinto', ax=axes[1])
plt.tight_layout(); plt.show()

# Mostrar camino aprendido
print('\nCamino aprendido en el laberinto:')
camino, _ = agente_maze.ejecutar_episodio()

# Dibujar camino en el grid
fig, ax = plt.subplots(figsize=(8, 8))
for i in range(8):
    for j in range(8):
        if (i,j) == env_maze.inicio:
            color = '#90EE90'
        elif (i,j) == env_maze.meta:
            color = '#FFD700'
        elif (i,j) in env_maze.obstaculos:
            color = '#2c3e50'
        elif (i,j) in camino:
            color = '#3498db'
        else:
            color = '#ecf0f1'
        rect = plt.Rectangle((j, 7-i), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)

# Flechas del camino
for k in range(len(camino)-1):
    r1, c1 = camino[k]
    r2, c2 = camino[k+1]
    ax.annotate('', xy=(c2+0.5, 7-r2+0.5), xytext=(c1+0.5, 7-r1+0.5),
               arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.set_xlim(0,8); ax.set_ylim(0,8); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(f'Camino aprendido ({len(camino)-1} pasos)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

agente_maze.plot_aprendizaje(ventana=200)

### Ejemplo 7: Efecto de los hiperparámetros

Exploremos cómo cada hiperparámetro afecta el aprendizaje.

In [ ]:
# Efecto de alpha (learning rate)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

env_hp = GridWorld(filas=5, columnas=5, inicio=(0,0), meta=(4,4),
                   obstaculos=[(1,1), (2,3), (3,1)], recompensa_meta=100)

# Alpha
alphas = [0.01, 0.1, 0.5, 0.9]
colores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
ax = axes[0]
for alpha, color in zip(alphas, colores):
    np.random.seed(42)
    a = AgenteRL(env_hp, alpha=alpha, gamma=0.99)
    a.entrenar(n_episodios=2000, verbose=False)
    v = 100
    media = np.convolve(a.recompensas_historial, np.ones(v)/v, mode='valid')
    ax.plot(media, label=f'α={alpha}', color=color, linewidth=2)
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa')
ax.set_title('Efecto de α (learning rate)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Gamma
gammas = [0.5, 0.9, 0.99, 0.999]
ax = axes[1]
for gam, color in zip(gammas, colores):
    np.random.seed(42)
    a = AgenteRL(env_hp, alpha=0.1, gamma=gam)
    a.entrenar(n_episodios=2000, verbose=False)
    media = np.convolve(a.recompensas_historial, np.ones(v)/v, mode='valid')
    ax.plot(media, label=f'γ={gam}', color=color, linewidth=2)
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa')
ax.set_title('Efecto de γ (factor de descuento)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Epsilon decay
decays = [0.99, 0.995, 0.999, 1.0]
labels_d = ['0.99 (rápido)', '0.995', '0.999 (lento)', '1.0 (constante)']
ax = axes[2]
for decay, label, color in zip(decays, labels_d, colores):
    np.random.seed(42)
    a = AgenteRL(env_hp, alpha=0.1, gamma=0.99, epsilon_decay=decay)
    a.entrenar(n_episodios=2000, verbose=False)
    media = np.convolve(a.recompensas_historial, np.ones(v)/v, mode='valid')
    ax.plot(media, label=f'decay={label}', color=color, linewidth=2)
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa')
ax.set_title('Efecto del decaimiento de ε', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Impacto de los hiperparámetros en el aprendizaje', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Observaciones:')
print('• α muy bajo (0.01): aprende lento pero estable')
print('• α muy alto (0.9): aprende rápido pero inestable')
print('• γ bajo (0.5): miópico, no planifica bien a largo plazo')
print('• γ alto (0.99): previsor, alcanza mejor rendimiento')
print('• ε decay rápido (0.99): deja de explorar pronto, puede quedar atrapado')
print('• ε decay lento (0.999): explora más tiempo, mejor cobertura')

### Ejemplo 8: Problema del Taxi

Un taxi debe recoger un pasajero en una ubicación y dejarlo en otra. Tiene acciones adicionales: recoger y dejar pasajero.

In [ ]:
class TaxiWorld:
    """Entorno Taxi simplificado en un grid 5x5."""
    
    ACCIONES = {0: (-1,0), 1: (1,0), 2: (0,-1), 3: (0,1), 4: 'recoger', 5: 'dejar'}
    FLECHAS = {0: '↑', 1: '↓', 2: '←', 3: '→', 4: 'P', 5: 'D'}
    N_ACCIONES = 6
    
    def __init__(self):
        self.filas = 5
        self.columnas = 5
        self.ubicaciones = [(0,0), (0,4), (4,0), (4,4)]  # R, G, Y, B
        self.nombres_ubic = ['R', 'G', 'Y', 'B']
        self.estado_actual = None
        self.pasajero_pos = None  # índice de ubicación o -1 si está en el taxi
        self.destino = None       # índice de ubicación destino
    
    def reset(self):
        self.estado_actual = (np.random.randint(5), np.random.randint(5))
        self.pasajero_pos = np.random.randint(4)
        self.destino = np.random.randint(4)
        while self.destino == self.pasajero_pos:
            self.destino = np.random.randint(4)
        return self._get_state()
    
    def _get_state(self):
        return (self.estado_actual[0], self.estado_actual[1], self.pasajero_pos, self.destino)
    
    def step(self, accion):
        recompensa = -1  # costo por paso
        terminado = False
        
        if accion < 4:  # movimiento
            dr, dc = self.ACCIONES[accion]
            new_r = max(0, min(4, self.estado_actual[0] + dr))
            new_c = max(0, min(4, self.estado_actual[1] + dc))
            self.estado_actual = (new_r, new_c)
        elif accion == 4:  # recoger
            if self.pasajero_pos >= 0 and self.estado_actual == self.ubicaciones[self.pasajero_pos]:
                self.pasajero_pos = -1  # pasajero en el taxi
            else:
                recompensa = -10  # recoger en lugar incorrecto
        elif accion == 5:  # dejar
            if self.pasajero_pos == -1 and self.estado_actual == self.ubicaciones[self.destino]:
                recompensa = 20  # éxito
                terminado = True
            else:
                recompensa = -10  # dejar en lugar incorrecto
        
        return self._get_state(), recompensa, terminado


# Q-Learning para Taxi (con tabla Q basada en diccionario)
np.random.seed(42)
taxi = TaxiWorld()

Q_taxi = {}  # diccionario: estado → array de Q-values
alpha, gamma = 0.1, 0.99
epsilon = 1.0
recompensas_taxi = []

for ep in range(10000):
    estado = taxi.reset()
    if estado not in Q_taxi:
        Q_taxi[estado] = np.zeros(6)
    
    r_total = 0
    for _ in range(200):
        if np.random.random() < epsilon:
            accion = np.random.randint(6)
        else:
            accion = np.argmax(Q_taxi[estado])
        
        nuevo_estado, r, done = taxi.step(accion)
        r_total += r
        
        if nuevo_estado not in Q_taxi:
            Q_taxi[nuevo_estado] = np.zeros(6)
        
        # Q-Learning
        target = r + gamma * np.max(Q_taxi[nuevo_estado]) * (1 - done)
        Q_taxi[estado][accion] += alpha * (target - Q_taxi[estado][accion])
        
        estado = nuevo_estado
        if done:
            break
    
    recompensas_taxi.append(r_total)
    epsilon = max(0.01, epsilon * 0.9995)

# Resultados
fig, ax = plt.subplots(figsize=(10, 5))
v = 200
media = np.convolve(recompensas_taxi, np.ones(v)/v, mode='valid')
ax.plot(media, color='#3498db', linewidth=2)
ax.set_xlabel('Episodio', fontsize=12); ax.set_ylabel('Recompensa', fontsize=12)
ax.set_title('Taxi: Curva de aprendizaje', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Demo de un episodio
print(f'Estados Q aprendidos: {len(Q_taxi)}')
print(f'Recompensa media (últimos 500): {np.mean(recompensas_taxi[-500:]):.1f}')

print('\n--- Demo de un episodio ---')
estado = taxi.reset()
print(f'Taxi en {taxi.estado_actual}, Pasajero en {taxi.nombres_ubic[taxi.pasajero_pos]}, '
      f'Destino: {taxi.nombres_ubic[taxi.destino]}')
for paso in range(20):
    if estado not in Q_taxi:
        break
    accion = np.argmax(Q_taxi[estado])
    estado, r, done = taxi.step(accion)
    nombre_a = ['↑','↓','←','→','Recoger','Dejar'][accion]
    print(f'  Paso {paso+1}: {nombre_a} → pos={taxi.estado_actual}, R={r:+d}', end='')
    if taxi.pasajero_pos == -1:
        print(' [pasajero en taxi]', end='')
    if done:
        print(' ¡ÉXITO!')
        break
    print()

### Ejemplo 9: Búsqueda del tesoro con recompensas intermedias

Un entorno donde hay tesoros pequeños en el camino al tesoro grande. El agente debe decidir si recolectarlos o ir directo a la meta.

In [ ]:
class TreasureHunt(GridWorld):
    """Grid con tesoros intermedios y tesoro principal."""
    
    def __init__(self):
        self.tesoros = {(0,3): 5, (2,1): 5, (1,4): 3, (4,2): 3}
        self.tesoros_recogidos = set()
        super().__init__(filas=6, columnas=6, inicio=(0,0), meta=(5,5),
                        obstaculos=[(2,2), (3,3), (1,1)],
                        recompensa_paso=-1, recompensa_obstaculo=-10,
                        recompensa_meta=50)
    
    def reset(self):
        self.tesoros_recogidos = set()
        return super().reset()
    
    def step(self, accion):
        nuevo_estado, recompensa, terminado = super().step(accion)
        # Bonus por tesoros
        if nuevo_estado in self.tesoros and nuevo_estado not in self.tesoros_recogidos:
            recompensa += self.tesoros[nuevo_estado]
            self.tesoros_recogidos.add(nuevo_estado)
        return nuevo_estado, recompensa, terminado


np.random.seed(42)
env_treasure = TreasureHunt()

# Entrenar con diferentes gammas
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
gammas_t = [0.5, 0.9, 0.99]

for ax, gam in zip(axes, gammas_t):
    np.random.seed(42)
    env_t = TreasureHunt()
    agente_t = AgenteRL(env_t, alpha=0.1, gamma=gam, epsilon_decay=0.999)
    agente_t.entrenar(n_episodios=5000, verbose=False)
    
    # Visualizar
    for i in range(6):
        for j in range(6):
            if (i,j) == env_t.inicio:
                color = '#90EE90'
            elif (i,j) == env_t.meta:
                color = '#FFD700'
            elif (i,j) in env_t.obstaculos:
                color = '#FF6B6B'
            elif (i,j) in env_t.tesoros:
                color = '#DDA0DD'
            else:
                color = '#E8E8E8'
            rect = plt.Rectangle((j, 5-i), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            if (i,j) == env_t.meta:
                ax.text(j+0.5, 5-i+0.5, '💎', ha='center', va='center', fontsize=14)
            elif (i,j) in env_t.tesoros:
                ax.text(j+0.5, 5-i+0.7, '🪙', ha='center', va='center', fontsize=10)
                ax.text(j+0.5, 5-i+0.3, f'+{env_t.tesoros[(i,j)]}', ha='center', va='center', fontsize=8)
            elif (i,j) in env_t.obstaculos:
                ax.text(j+0.5, 5-i+0.5, 'X', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
            elif (i,j) != env_t.inicio:
                best_a = np.argmax(agente_t.Q[i,j])
                ax.text(j+0.5, 5-i+0.5, env_t.FLECHAS[best_a], ha='center', va='center', fontsize=16)
    
    ax.set_xlim(0,6); ax.set_ylim(0,6); ax.set_aspect('equal'); ax.axis('off')
    rew = np.mean(agente_t.recompensas_historial[-200:])
    ax.set_title(f'γ={gam} (R media={rew:.0f})', fontsize=12, fontweight='bold')

plt.suptitle('Búsqueda del tesoro: ¿recoger tesoros intermedios o ir directo?',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Observaciones:')
print('• γ=0.5 (miópico): va directo al tesoro más cercano o a la meta')
print('• γ=0.9: buen balance, recoge algunos tesoros en el camino')
print('• γ=0.99 (previsor): planifica rutas que maximizan recompensa total')

### Ejemplo 10: Windy GridWorld (viento cruzado)

Un GridWorld donde hay viento que empuja al agente hacia arriba en ciertas columnas. El agente debe compensar el viento para llegar a la meta.

In [ ]:
class WindyGridWorld(GridWorld):
    """GridWorld con viento cruzado que empuja al agente hacia arriba."""
    
    def __init__(self):
        super().__init__(filas=7, columnas=10, inicio=(3,0), meta=(3,7),
                        recompensa_paso=-1, recompensa_meta=0)
        # Fuerza del viento por columna (empuja hacia arriba)
        self.viento = [0, 0, 0, 1, 1, 1, 2, 2, 1, 0]
    
    def step(self, accion):
        dr, dc = self.ACCIONES[accion]
        col_actual = self.estado_actual[1]
        
        # Aplicar movimiento + viento
        nueva_fila = self.estado_actual[0] + dr - self.viento[col_actual]
        nueva_col = self.estado_actual[1] + dc
        
        nueva_fila = max(0, min(self.filas - 1, nueva_fila))
        nueva_col = max(0, min(self.columnas - 1, nueva_col))
        
        nuevo_estado = (nueva_fila, nueva_col)
        recompensa = self.recompensas[nuevo_estado]
        terminado = (nuevo_estado == self.meta)
        
        self.estado_actual = nuevo_estado
        return nuevo_estado, recompensa, terminado


np.random.seed(42)
env_windy = WindyGridWorld()

# Entrenar
agente_windy = AgenteRL(env_windy, algoritmo='q_learning', alpha=0.5, gamma=0.99,
                         epsilon_inicio=0.1, epsilon_fin=0.1, epsilon_decay=1.0)
agente_windy.entrenar(n_episodios=2000)

# Visualizar
fig, ax = plt.subplots(figsize=(14, 6))
for i in range(7):
    for j in range(10):
        wind_strength = env_windy.viento[j]
        if (i,j) == env_windy.inicio:
            color = '#90EE90'
        elif (i,j) == env_windy.meta:
            color = '#FFD700'
        elif wind_strength == 0:
            color = '#E8E8E8'
        elif wind_strength == 1:
            color = '#B3D9FF'
        else:
            color = '#66B2FF'
        
        rect = plt.Rectangle((j, 6-i), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        
        if (i,j) == env_windy.meta:
            ax.text(j+0.5, 6-i+0.5, 'G', ha='center', va='center', fontsize=12, fontweight='bold')
        elif (i,j) == env_windy.inicio:
            ax.text(j+0.5, 6-i+0.5, 'S', ha='center', va='center', fontsize=12, fontweight='bold')
        else:
            best_a = np.argmax(agente_windy.Q[i,j])
            ax.text(j+0.5, 6-i+0.5, env_windy.FLECHAS[best_a], ha='center', va='center', fontsize=14)

# Mostrar fuerza del viento debajo
for j in range(10):
    w = env_windy.viento[j]
    ax.text(j+0.5, -0.3, f'↑{w}' if w > 0 else '0', ha='center', va='center',
           fontsize=11, fontweight='bold', color='#2980b9' if w > 0 else 'gray')
ax.text(5, -0.8, 'Fuerza del viento por columna', ha='center', fontsize=12, fontstyle='italic')

ax.set_xlim(0,10); ax.set_ylim(-1.2,7); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Windy GridWorld: el agente aprende a compensar el viento', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Mostrar camino
print('Camino aprendido con viento:')
camino_w, _ = agente_windy.ejecutar_episodio()

agente_windy.plot_aprendizaje(ventana=100)

print('\nEl agente aprende a ir hacia ABAJO en columnas con viento fuerte')
print('para compensar el empuje hacia arriba.')

---

## Resumen

| Concepto | Descripción |
|---|---|
| **Objetivo** | Aprender una política que maximice la recompensa acumulada |
| **Marco formal** | MDP: (S, A, P, R, γ) |
| **Ecuación clave** | Bellman: V(s) = R + γ·V(s') |
| **Q-Learning** | Off-policy: Q(s,a) ← Q(s,a) + α[R + γ·max Q(s',a') - Q(s,a)] |
| **SARSA** | On-policy: Q(s,a) ← Q(s,a) + α[R + γ·Q(s',a') - Q(s,a)] |
| **Exploración** | ε-greedy con decaimiento |
| **α (learning rate)** | Qué tanto actualizamos (0.1-0.5 típico) |
| **γ (descuento)** | Cuánto valoramos el futuro (0.9-0.99 típico) |
| **ε (exploración)** | Probabilidad de acción aleatoria (1→0.01 con decay) |
| **Q-Learning vs SARSA** | Q-Learning: óptimo pero agresivo. SARSA: conservador pero seguro |

### Ejemplos cubiertos

| # | Ejemplo | Concepto demostrado |
|---|---|---|
| 1 | GridWorld 4x4 | Q-Learning básico |
| 2 | Q-Learning vs SARSA | Comparación de algoritmos |
| 3 | Cliff Walking | Off-policy vs On-policy (seguridad) |
| 4 | Frozen Lake | Entornos estocásticos |
| 5 | Multi-Armed Bandit | Exploración vs explotación pura |
| 6 | Laberinto 8x8 | Problemas más complejos |
| 7 | Hiperparámetros | Efecto de α, γ, ε |
| 8 | Taxi | Espacios de estado grandes, acciones adicionales |
| 9 | Búsqueda del tesoro | Recompensas intermedias, efecto de γ |
| 10 | Windy GridWorld | Dinámica del entorno, compensar perturbaciones |